# BÁO CÁO THÍ NGHIỆM LAB 3: NHẬN DẠNG PHƯƠNG TIỆN VÀ QUY ĐỔI TẢI TRỌNG PCU
**Học phần:** Xử lý ảnh và Thị giác máy tính (121036) - Trường Đại học Giao thông vận tải TP.HCM (UTH)  
**Thành viên phụ trách:** Thành viên 5 (TV5)  
**Mục tiêu:** Nhận dạng, phân loại chính xác 4 nhóm phương tiện giao thông đường bộ (Xe máy, Ô tô con, Xe buýt, Xe tải), quy đổi lưu lượng thực tế sang hệ số Đơn vị Xe con Tiêu chuẩn (Passenger Car Unit - PCU), và thực hiện khảo sát tham số (Parameter Sweep) tối ưu hóa khả năng tách biệt giữa xe buýt và xe tải thùng dài.

---
## 1. Cơ Sở Lý Thuyết và Quy Chuẩn Đơn Vị Xe Con Tiêu Chuẩn (PCU)
Trong bài toán đánh giá ùn tắc giao thông tại đô thị Việt Nam (đặc biệt các tuyến đường vành đai và nút giao trọng điểm tại TP.HCM), mỗi phương tiện có diện tích chiếm dụng mặt đường và tính năng động học khác nhau.
Theo **Quy chuẩn Kỹ thuật Quốc gia về Giao thông Đô thị**, hệ số quy đổi PCU được thiết lập:
- **Xe máy (motorcycle):** 0.33 PCU (khoảng 3 xe máy tương đương diện tích và độ cản trở của 1 ô tô con).
- **Ô tô con / Taxi (car):** 1.0 PCU (Đơn vị tiêu chuẩn cơ sở).
- **Xe buýt (bus):** 2.5 PCU (Phương tiện công cộng thân dài, thường xuyên dừng đón trả khách).
- **Xe tải (truck):** 3.0 PCU (Phương tiện tải trọng lớn, quán tính cao, cản trở đáng kể dòng lưu thông).

Tổng tải trọng quy đổi được tính theo công thức:
$$\text{Total PCU} = \sum_{i \in \text{Classes}} N_i \times \text{PCU}_i$$
Chỉ số này sau đó được chuẩn hóa thành mật độ tải trọng $D_{\text{PCU}} = \min\left(\frac{\text{Total PCU}}{\text{Max Capacity}}, 1.0\right)$ gửi tới TV1 để tính toán Chỉ số Ùn tắc Tổng hợp (TCI).

In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Thêm đường dẫn gốc dự án
sys.path.append('..')
from config import PCU_WEIGHTS
from modules.detection import VehicleDetector

print('Cấu hình trọng số PCU chuẩn hóa:')
for vehicle, pcu in PCU_WEIGHTS.items():
    print(f'  - {vehicle:12s}: {pcu:.2f} PCU')


---
## 2. Khởi Tạo Mô Hình YOLOv8 và Kiểm Tra Frame Giao Thông
Module `VehicleDetector` ánh xạ 4 nhóm phương tiện từ tập dữ liệu COCO chuẩn:
- Class 2: `car`
- Class 3: `motorcycle`
- Class 5: `bus`
- Class 7: `truck`

In [ ]:
# Khởi tạo detector với ngưỡng tin cậy mặc định 0.4
detector = VehicleDetector(conf_thresh=0.4, iou_thresh=0.45)

video_path = os.path.join('..', 'data', 'raw', 'traffic_congested.mp4')
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_MSEC, 3000) # Lấy frame tại giây thứ 3
ret, sample_frame = cap.read()
cap.release()

if ret:
    print(f'Đọc thành công frame kích thước: {sample_frame.shape}')
    total_pcu, counts, detections = detector.detect_and_count_pcu(sample_frame, return_counts=True)
    print(f'Tổng số phương tiện phát hiện: {len(detections)}')
    print(f'Thống kê số lượng: {counts}')
    print(f'Tổng tải trọng quy đổi: {total_pcu:.2f} PCU')
else:
    print('Không đọc được video, sử dụng dữ liệu mô phỏng.')


---
## 3. Trực Quan Hóa Bounding Box và Phân Bổ Tải Trọng
Vẽ các khung nhận diện với hệ màu quy chuẩn:
- **Xanh ngọc (Cyan):** Xe máy
- **Xanh lá (Green):** Ô tô con
- **Cam (Orange):** Xe buýt
- **Đỏ (Red):** Xe tải lớn

In [ ]:
if ret:
    annotated = detector.draw_detections(sample_frame, detections, draw_hud=True)
    rgb_frame = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Hiển thị frame nhận diện
    ax1.imshow(rgb_frame)
    ax1.set_title('Kết Quả Nhận Dạng Phương Tiện Bounding Box (YOLOv8)', fontsize=12)
    ax1.axis('off')
    
    # Biểu đồ đóng góp PCU theo loại xe
    classes = list(counts.keys())
    pcu_contribs = [counts[c] * PCU_WEIGHTS.get(c, 1.0) for c in classes]
    colors = ['#00d4ff', '#2ecc71', '#ff9f43', '#ee5253']
    
    bars = ax2.bar(classes, pcu_contribs, color=colors, edgecolor='black', alpha=0.85)
    ax2.set_title('Đóng Góp Tải Trọng PCU Theo Từng Nhóm Phương Tiện', fontsize=12)
    ax2.set_ylabel('Tổng PCU', fontsize=11)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax2.text(bar.get_x() + bar.get_width()/2., h + 0.3, f'{h:.1f}', ha='center', va='bottom', fontweight='bold')
            
    plt.tight_layout()
    plt.show()


---
## 4. Khảo Sát Tham Số (Parameter Sweep): Phân Biệt Xe Buýt vs Xe Tải
**Bài toán đặt ra:** Xe buýt và xe tải thùng dài có hình dạng chữ nhật kéo dài tương đồng nhau từ góc nhìn camera nghiêng (CCTV).
Khi ngưỡng `conf_threshold` quá thấp ($< 0.35$), mô hình thường sinh ra 2 bounding box trùng lặp nhau cho cùng 1 xe (vừa dự đoán là Bus, vừa dự đoán là Truck).
Ta tiến hành khảo sát lưới tham số:
- `conf_threshold` $\in [0.25, 0.40, 0.55, 0.70]$
- `iou_threshold` $\in [0.30, 0.45, 0.60]$

In [ ]:
if ret:
    conf_list = [0.25, 0.40, 0.55, 0.70]
    iou_list = [0.30, 0.45, 0.60]
    sweep_results = detector.sweep_parameters(sample_frame, conf_thresholds=conf_list, iou_thresholds=iou_list)
    
    import pandas as pd
    df_sweep = pd.DataFrame(sweep_results)
    print('BẢNG KẾT QUẢ KHẢO SÁT THAM SỐ:')
    display(df_sweep) if 'display' in dir() else print(df_sweep.to_string())


In [ ]:
# Biểu đồ phân tích độ nhạy của ngưỡng Confidence
if ret:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    for iou in [0.30, 0.45, 0.60]:
        sub_df = [r for r in sweep_results if r['iou_threshold'] == iou]
        confs = [r['conf_threshold'] for r in sub_df]
        truck_counts = [r['truck'] for r in sub_df]
        bus_counts = [r['bus'] for r in sub_df]
        pcu_vals = [r['total_pcu'] for r in sub_df]
        
        ax1.plot(confs, truck_counts, marker='s', linestyle='-', label=f'Truck (IoU={iou})')
        ax1.plot(confs, bus_counts, marker='o', linestyle='--', label=f'Bus (IoU={iou})')
        ax2.plot(confs, pcu_vals, marker='^', label=f'Total PCU (IoU={iou})')
        
    ax1.set_title('Biến Thiên Số Lượng Xe Tải và Xe Buýt Theo Confidence', fontsize=12)
    ax1.set_xlabel('Confidence Threshold', fontsize=11)
    ax1.set_ylabel('Số lượng phát hiện', fontsize=11)
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend()
    
    ax2.set_title('Tổng Chỉ Số PCU Theo Confidence và IoU Threshold', fontsize=12)
    ax2.set_xlabel('Confidence Threshold', fontsize=11)
    ax2.set_ylabel('Total PCU', fontsize=11)
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()


---
## 5. Kết Luận và Khuyến Nghị Thực Nghiệm Cho Hệ Thống
1. **Ngưỡng Confidence tối ưu:**
   - Khi $\text{conf} = 0.25$, mô hình phát hiện được xe ở xa nhưng bị hiện tượng nhận diện nhầm đúp (cùng 1 thân xe dài bị gán cả nhãn Bus và Truck).
   - Khi $\text{conf} \ge 0.70$, các phương tiện ở xa hoặc bị che khuất một phần bị bỏ sót hoàn toàn, làm suy giảm tổng PCU thực tế.
   - **Khuyến nghị:** Chọn $\text{conf\_thresh} \in [0.40, 0.45]$ kết hợp $\text{iou\_thresh} = 0.45$. Cấu hình này giúp phân biệt rành mạch giữa xe buýt và xe tải, loại bỏ box trùng và duy trì độ ổn định tải trọng.

2. **Đóng góp vào Pipeline chung:**
   Chỉ số `total_pcu` cung cấp thông tin tải trọng giao thông chính xác cho TV1, phản ánh đúng thực tế giao thông Việt Nam khi xe tải (3.0 PCU) và xe buýt (2.5 PCU) có sức cản trở cao hơn nhiều so với ô tô con và xe máy.